### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
        unique_name="mercari_price_suggestion_1m",
    version_from_unique_name="mercari_price_suggestion",
    version_comment="""
We build the recommended single train/test split (250k test rows) and sub-sample it to at most 1M train and 250k test rows with `curation_recommendations.subsample_split_to_budget`.
""",
    # Same as the full-size `mercari_price_suggestion` dataset
    dataset_year="2018",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/competitions/mercari-price-suggestion-challenge",
    download_description="""
train.tsv.7z

kaggle competitions download -c mercari-price-suggestion-challenge -f train.tsv.7z && 7z x train.tsv.7z && rm train.tsv.7z
mkdir -p local-data-warehouse/mercari_price_suggestion && mv train.tsv local-data-warehouse/mercari_price_suggestion
""",
    # References
    academic_reference_bibtex=r"""@misc{Howard2017MercariPriceSuggestionChallenge,
  author = {{Kaggle} and Addison Howard and kaoriiida and Kei Otagaki and Mark McDonald and mueno and Wendy Kan and Zhang and zyaga},
  title  = {Mercari Price Suggestion Challenge},
  year   = {2017},
  howpublished = {\url{https://kaggle.com/competitions/mercari-price-suggestion-challenge}},
  note   = {Kaggle competition}
}
""",
    academic_reference_bibtex_key="Howard2017MercariPriceSuggestionChallenge",
    license="Kaggle Competition Rules",
    data_tags=["IID"],
    curation_comments="""
We start from the Kaggle competition dataset.

- We log1p scale the target to match the target metric of the competition by default RMSE.
- The dataset was solved by Kaggle experts by tabular models and a lot of custom string preprocessing. So it is a great use case to test encoding pipelines! The dataset also has mostly string data.
- The task was treated and solved as an IID task on Kaggle.
- We replace "No description yet" with NaN.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="price",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "train.tsv", sep="\t")

df["price"] = np.log1p(df["price"])
df = df.drop(columns=["train_id"])
df["item_description"] = df["item_description"].replace("No description yet", np.nan)

as_cat_dtype = [
    "item_condition_id",
    "shipping",
]
as_string_dtype = [
    "name",
    "category_name", # multi-categorical, not just one category name!
    "brand_name",
    "item_description",
]

for c in as_string_dtype:
    nan_mask = df[c].isna()
    df.loc[nan_mask, c] = np.nan
    df[c] = df[c].astype("string")

df[as_cat_dtype] = df[as_cat_dtype].astype("category")

# Drop duplicates
df = df.drop_duplicates()

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
    duplicate_column_check=False, # skip, no duplicates
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)

# -- For IID Data
splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    random_state=split_random_state,
)

# Sub-sample the single train/test split to the row budget (1M train / 250k test rows).
train_idx, test_idx = splits[0][0]
df, train_idx, test_idx = curation_recommendations.subsample_split_to_budget(
    df=df,
    train_idx=train_idx,
    test_idx=test_idx,
    group_on=task_mold.group_on,
    stratify_on=task_mold.stratify_on,
    random_state=split_random_state,
)
splits = {0: {0: (train_idx, test_idx)}}
print(f"Final train size: {len(train_idx):,}, final test size: {len(test_idx):,}")

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Recommended single train/test split, sub-sampled to the row budget of 1M train and 250k test rows.",
    splits=splits,
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()